# 01 — Tu primer stream: `rate` source + windowing

Empezamos con la fuente más simple que existe: una que genera filas
automáticamente a un ritmo configurable.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("03_01_rate_windowing")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
# Genera N filas por segundo, con un id y un timestamp
rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)
rate_stream.printSchema()

## Tumbling window: contar eventos por minuto

In [ ]:
windowed_counts = (
    rate_stream
    .groupBy(
        F.window(F.col("timestamp"), "1 minute")
    )
    .count()
    .select(
        F.col("window.start").alias("inicio"),
        F.col("window.end").alias("fin"),
        F.col("count"),
    )
)
windowed_counts.printSchema()

## Sink a memoria (debug)

Para experimentar rápido, escribimos a una tabla en memoria. En
producción escribirías a Delta, Kafka, o consola.

In [ ]:
query = (
    windowed_counts
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("eventos_por_minuto")
    .start()
)
print("Query ID:", query.id)
print("Status:", query.status)

In [ ]:
import time
time.sleep(10)  # dejamos correr 10s
spark.sql("SELECT * FROM eventos_por_minuto ORDER BY inicio").show(truncate=False)

In [ ]:
query.stop()
print("Query detenido")

## Sliding window

Ventana de 1 minuto que avanza cada 30 segundos. Útil para
visualizaciones suaves.

In [ ]:
sliding = (
    rate_stream
    .groupBy(
        F.window(F.col("timestamp"), "1 minute", "30 seconds")
    )
    .count()
)
sliding.printSchema()

## ✅ Checkpoint

- [x] Sé levantar un `readStream` con `rate` source.
- [x] Sé agrupar con `window(...)` (tumbling y sliding).
- [x] Sé escribir a memoria para debug.

**Siguiente**: `02_watermarks_late_data.ipynb` — qué pasa cuando los
eventos llegan tarde.